# Fluxonium module preview

Run this notebook inside the `qmetal` miniforge environment.  It instantiates the redesigned `FluxoniumModule` and shows the Qiskit Metal geometry, including readout, charge, flux, and FTF transmon-coupler pins.

In [5]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

workspace = Path.cwd()
if str(workspace) not in sys.path:
    sys.path.insert(0, str(workspace))

from qiskit_metal import designs
from custom.fluxonium_module import FluxoniumModule

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

fluxonium = FluxoniumModule(
    design,
    "F0",
    options=dict(
        pos_x="0um",
        pos_y="0um",
        orientation="0",
        readout_line_options=dict(make=True),
        charge_line_options=dict(make=True),
        flux_line_options=dict(make=True),
        transmon_coupler_options=dict(make=True),
    ),
)

print("components:", list(design.components.keys()))
print("pins:", sorted(fluxonium.pins.keys()))
print("qgeometry rows:", {name: len(table) for name, table in design.qgeometry.tables.items()})

components: ['F0']
pins: ['charge', 'flux', 'readout', 'transmon_coupler']
qgeometry rows: {'path': 24, 'poly': 10, 'junction': 2}


In [7]:
# Qiskit Metal native GUI.  This is the best view when running locally.
from qiskit_metal import MetalGUI

gui = MetalGUI(design)
gui.rebuild()
gui.autoscale()

In [8]:
# Static matplotlib fallback for quick notebook inspection.
import matplotlib.pyplot as plt
from shapely.geometry import LineString, MultiLineString, MultiPolygon, Polygon

def plot_geom(ax, geom, **kwargs):
    if isinstance(geom, Polygon):
        x, y = geom.exterior.xy
        ax.fill(x, y, **kwargs)
    elif isinstance(geom, MultiPolygon):
        for part in geom.geoms:
            plot_geom(ax, part, **kwargs)
    elif isinstance(geom, LineString):
        x, y = geom.xy
        ax.plot(x, y, **kwargs)
    elif isinstance(geom, MultiLineString):
        for part in geom.geoms:
            plot_geom(ax, part, **kwargs)

fig, ax = plt.subplots(figsize=(9, 7))

for _, row in design.qgeometry.tables["poly"].iterrows():
    color = "#b7d7e8" if not row.get("subtract", False) else "#eeeeee"
    alpha = 0.85 if not row.get("subtract", False) else 0.35
    plot_geom(ax, row.geometry, color=color, alpha=alpha, linewidth=0.8)

for _, row in design.qgeometry.tables["path"].iterrows():
    color = "#1f77b4" if not row.get("subtract", False) else "#cccccc"
    plot_geom(ax, row.geometry, color=color, linewidth=2)

for _, row in design.qgeometry.tables["junction"].iterrows():
    plot_geom(ax, row.geometry, color="#d62728", linewidth=2.5)

ax.set_aspect("equal", adjustable="box")
ax.set_title("FluxoniumModule QGeometry preview")
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.grid(True, linewidth=0.3, alpha=0.4)
plt.show()